In [1]:
!git clone https://github.com/Erick-FHP/broad-money-latam-analysis.git

Cloning into 'broad-money-latam-analysis'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 4.84 KiB | 2.42 MiB/s, done.


# Cargando los datos crudos

In [40]:
import pandas as pd

PATH_CSV = './broad-money-latam-analysis/data/raw/data_broad_money.csv'

df = pd.read_csv(PATH_CSV)
df.drop(columns=['Country Name', 'Series Name', 'Series Code'], inplace=True)
df_long = (
    df.set_index("Country Code")      # o el nombre real de tu columna de países
      .stack()
      .reset_index(name="broad_money")
      .rename(columns={"level_1": "year", "Country Code": "Country"})
)
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Country      300 non-null    object
 1   year         300 non-null    object
 2   broad_money  300 non-null    object
dtypes: object(3)
memory usage: 7.2+ KB


In [41]:
df_long.head()

,Country,year,broad_money
0,MEX,1976 [YR1976],38.732312
1,MEX,1977 [YR1977],29.978367
2,MEX,1978 [YR1978],32.023962
3,MEX,1979 [YR1979],33.132334
4,MEX,1980 [YR1980],29.776192


In [42]:
df_long['year'] = df_long['year'].str.split().str[0].astype(int)
df_long['broad_money'] = pd.to_numeric(df_long['broad_money'], errors='coerce')
df_long['broad_money'] = df_long['broad_money'].ffill()
df_long.head()

,Country,year,broad_money
0,MEX,1976,38.732312
1,MEX,1977,29.978367
2,MEX,1978,32.023962
3,MEX,1979,33.132334
4,MEX,1980,29.776192


In [43]:
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      300 non-null    object 
 1   year         300 non-null    int64  
 2   broad_money  300 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 7.2+ KB


In [44]:
df_long['Country'].value_counts()

,count
Country,
MEX,50
COL,50
BRA,50
WLD,50
URY,50
CHL,50


# Creando nuevas columnas

In [45]:
df_long['decade'] = (df_long['year'] // 10) * 10
df_long['annual change'] = df_long.groupby('Country')['broad_money'].diff()

world = df_long[df_long['Country'] == 'WLD'][['year', 'broad_money']].rename(columns={"broad_money": "world_money"})
df_long = df_long.merge(
    world,
    on="year",
    how="left"
)
df_long["gap_world"] = (
    df_long["broad_money"] -
    df_long["world_money"]
)
df_long["gap_world_pct"] = (
    (df_long["broad_money"] - df_long["world_money"])
    / df_long["world_money"]
    * 100
)
df_long = (
    df_long
    .sort_values(["Country", "year"])
    .reset_index(drop=True)
)
df_long

,Country,year,broad_money,decade,annual change,world_money,gap_world,gap_world_pct
0,BRA,1976,15.300223,1970,NaN,61.832874,-46.532651,-75.255520
1,BRA,1977,14.520787,1970,-0.779437,62.126561,-47.605774,-76.627088
2,BRA,1978,14.569096,1970,0.048310,65.392679,-50.823583,-77.720601
3,BRA,1979,15.298841,1970,0.729744,64.114462,-48.815621,-76.138237
4,BRA,1980,11.768651,1980,-3.530190,62.948836,-51.180185,-81.304419
...,...,...,...,...,...,...,...,...
295,WLD,2021,139.915617,2020,-0.167703,139.915617,0.000000,0.000000
296,WLD,2022,136.903837,2020,-3.011780,136.903837,0.000000,0.000000
297,WLD,2023,135.565364,2020,-1.338473,135.565364,0.000000,0.000000
298,WLD,2024,136.105562,2020,0.540198,136.105562,0.000000,0.000000


# Exportando el dataset final

In [46]:
df_long.to_csv(
    "./broad_money_clean.csv",
    index=False
)